# Task 2 — Gym SQL queries

Keep `gym.sqlite3` beside this notebook and select **Run All** (SQLite 3.37+). Amounts are in KZT; timestamps are UTC. DELETE/UPDATE examples roll back automatically.


In [1]:
from pathlib import Path
from contextlib import contextmanager
from html import escape
import sqlite3
from IPython.display import display, HTML, Markdown

if sqlite3.sqlite_version_info < (3, 37, 0):
    raise RuntimeError("Use a Python kernel with SQLite 3.37+ for STRICT tables.")
if "conn" in globals():
    conn.close()
# Supports kernels launched in either this folder or the pda project root.
candidates = [Path("gym.sqlite3"), Path("gym-database/gym.sqlite3")]
db_path = next((p.resolve() for p in candidates if p.is_file()), None)
if db_path is None:
    raise FileNotFoundError("Keep gym.sqlite3 beside the notebook, then run from that folder.")
conn = sqlite3.connect(db_path.as_uri() + "?mode=rw", uri=True, isolation_level=None)
conn.execute("PRAGMA foreign_keys = ON")
assert conn.execute("PRAGMA foreign_keys").fetchone()[0] == 1
result_counts = {}

def display_table(columns, rows):
    def format_value(column, value):
        if value is None:
            return "—"
        if column.endswith("_kzt"):
            return f"{value:,.2f}"
        return str(value)
    header = "".join(f"<th>{escape(column)}</th>" for column in columns)
    body = "".join("<tr>" + "".join(
        f"<td>{escape(format_value(column, value))}</td>"
        for column, value in zip(columns, row)
    ) + "</tr>" for row in rows)
    display(HTML("""<style>
    .gym-results {border-collapse:collapse; font-family:system-ui; font-size:14px;}
    .gym-results th {background:#17324d; color:white; padding:9px; text-align:left;}
    .gym-results td {padding:8px; border-bottom:1px solid #dde3e9; color:#172b3a;}
    .gym-results tbody tr:nth-child(odd) {background:#f2f6fa;}
    .gym-results tbody tr:nth-child(even) {background:#fff;}
    </style>""" + f'<div style="overflow-x:auto"><table class="gym-results"><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table></div>'))

def show_query(label, sql):
    display(Markdown("```sql\n" + sql.strip() + "\n```"))
    cursor = conn.execute(sql)
    columns = [column[0] for column in cursor.description]
    rows = cursor.fetchall()
    cursor.close()
    display_table(columns, rows)
    result_counts[label] = len(rows)
    display(Markdown(f"**{len(rows)} row(s)** returned."))

@contextmanager
def rollback_demo():
    conn.execute("SAVEPOINT assignment_demo")
    try:
        yield
    finally:
        conn.execute("ROLLBACK TO SAVEPOINT assignment_demo")
        conn.execute("RELEASE SAVEPOINT assignment_demo")

print(f"Connected to {db_path.name} with SQLite {sqlite3.sqlite_version}. Foreign keys enabled.")


Connected to gym.sqlite3 with SQLite 3.46.1. Foreign keys enabled.


## Data


In [2]:
tables = ["members", "member_profiles", "membership_plans", "memberships",
          "trainers", "gym_classes", "class_bookings"]
# The identifiers below are a fixed allowlist, not user input.
counts = [(table, conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0])
          for table in tables]
display_table(["table", "rows"], counts)


table,rows
members,100
member_profiles,69
membership_plans,10
memberships,115
trainers,14
gym_classes,40
class_bookings,331


## 1.1 · Simple query — member directory


In [3]:
show_query("1.1", """SELECT member_id, first_name, last_name, email, phone
FROM members;""")


```sql
SELECT member_id, first_name, last_name, email, phone
FROM members;
```

member_id,first_name,last_name,email,phone
1,Aida,Karimova,aida@example.test,+77010000001
2,Daniyar,Omarov,daniyar@example.test,+77010000002
6,Alina,Smailova,alina@example.test,—
7,Timur,Bekov,timur@example.test,+77010000004
8,Madina,Aliyeva,madina@example.test,+77010000005
9,Ruslan,Kim,ruslan@example.test,—
10,Dana,Lee,dana@example.test,+77010000006
11,Arman,Sultanov,arman@example.test,+77010000007
12,Kamila,Nur,kamila@example.test,—
13,Olzhas,Serik,olzhas@example.test,+77010000008


**100 row(s)** returned.

## 1.2 · Simple query — active plans


In [4]:
show_query("1.2", """SELECT plan_name, duration_days, price_kzt
FROM membership_plans
WHERE is_active = true;""")


```sql
SELECT plan_name, duration_days, price_kzt
FROM membership_plans
WHERE is_active = true;
```

plan_name,duration_days,price_kzt
Monthly,30,"20,000.00"
Quarterly,90,"54,000.00"
Student,30,"12,000.00"
Premium,365,"180,000.00"
Weekend,30,"20,000.00"
Family,90,"54,000.00"
Youth,30,"12,000.00"
Annual Plus,365,"180,000.00"


**8 row(s)** returned.

## 2.1 · Two-table JOIN — emergency contacts


In [5]:
show_query("2.1", """SELECT m.first_name, m.last_name,
       p.emergency_contact_name, p.emergency_contact_phone
FROM members AS m
JOIN member_profiles AS p ON p.member_id = m.member_id;""")


```sql
SELECT m.first_name, m.last_name,
       p.emergency_contact_name, p.emergency_contact_phone
FROM members AS m
JOIN member_profiles AS p ON p.member_id = m.member_id;
```

first_name,last_name,emergency_contact_name,emergency_contact_phone
Aida,Karimova,Aliya Karimova,+77010000003
Alina,Smailova,Demo contact alina,+77019990000
Timur,Bekov,Demo contact timur,+77019990000
Madina,Aliyeva,Demo contact madina,+77019990000
Dana,Lee,Demo contact dana,+77019990000
Kamila,Nur,Demo contact kamila,+77019990000
Aruzhan,Beketova,Contact 1 for additional demo member,+77029990000
Zarina,Saparova,Contact 2 for additional demo member,+77029990001
Dias,Tulegenov,Contact 3 for additional demo member,+77029990002
Amina,Iskakova,Contact 4 for additional demo member,+77029990003


**69 row(s)** returned.

## 2.2 · Two-table JOIN — class instructors


In [6]:
show_query("2.2", """SELECT c.class_name, c.starts_at,
       t.first_name AS trainer_first_name, t.last_name AS trainer_last_name
FROM gym_classes AS c
JOIN trainers AS t ON t.trainer_id = c.trainer_id;""")


```sql
SELECT c.class_name, c.starts_at,
       t.first_name AS trainer_first_name, t.last_name AS trainer_last_name
FROM gym_classes AS c
JOIN trainers AS t ON t.trainer_id = c.trainer_id;
```

class_name,starts_at,trainer_first_name,trainer_last_name
Morning Strength,2026-11-20T04:00:00Z,Askar,Sadykov
Strength Basics,2026-11-10T04:00:00Z,Askar,Sadykov
Evening Strength,2026-11-21T13:00:00Z,Askar,Sadykov
Morning Stretch,2026-11-09T03:00:00Z,Aigerim,Tursyn
Yoga Flow,2026-11-22T05:00:00Z,Aigerim,Tursyn
Cardio Circuit,2026-11-24T12:00:00Z,Marat,Ivanov
Boxing Basics,2026-11-23T12:00:00Z,Marat,Ivanov
Pilates Intro,2026-11-25T07:00:00Z,Elena,Petrova
Functional Fitness,2026-11-20T04:00:00Z,Dinara,Abenova
Power Training,2026-11-10T04:00:00Z,Dinara,Abenova


**40 row(s)** returned.

## 3.1 · Three-table JOIN — purchased memberships


In [7]:
show_query("3.1", """SELECT m.first_name, m.last_name, p.plan_name,
       ms.starts_on, ms.ends_on, ms.paid_amount_kzt
FROM members AS m
JOIN memberships AS ms ON ms.member_id = m.member_id
JOIN membership_plans AS p ON p.membership_plan_id = ms.membership_plan_id;""")


```sql
SELECT m.first_name, m.last_name, p.plan_name,
       ms.starts_on, ms.ends_on, ms.paid_amount_kzt
FROM members AS m
JOIN memberships AS ms ON ms.member_id = m.member_id
JOIN membership_plans AS p ON p.membership_plan_id = ms.membership_plan_id;
```

first_name,last_name,plan_name,starts_on,ends_on,paid_amount_kzt
Aida,Karimova,Monthly,2026-11-01,2026-12-01,"18,000.00"
Olzhas,Serik,Monthly,2026-09-01,2026-10-01,0.00
Kamila,Nur,Monthly,2026-10-01,2026-10-31,"20,000.00"
Madina,Aliyeva,Monthly,2026-11-01,2026-12-01,"20,000.00"
Aida,Karimova,Monthly,2026-09-01,2026-10-01,"20,000.00"
Aida,Karimova,Monthly,2026-10-01,2026-10-31,"20,000.00"
Kamila,Nur,Quarterly,2026-11-01,2027-01-30,"54,000.00"
Dana,Lee,Quarterly,2026-11-01,2027-01-30,"50,000.00"
Daniyar,Omarov,Quarterly,2026-10-01,2026-12-30,"54,000.00"
Sabina,Khan,Student,2026-09-01,2026-10-01,"12,000.00"


**115 row(s)** returned.

## 3.2 · Three-table JOIN — class bookings


In [8]:
show_query("3.2", """SELECT m.first_name, m.last_name, c.class_name,
       c.starts_at, b.booking_status
FROM members AS m
JOIN class_bookings AS b ON b.member_id = m.member_id
JOIN gym_classes AS c ON c.gym_class_id = b.gym_class_id;""")


```sql
SELECT m.first_name, m.last_name, c.class_name,
       c.starts_at, b.booking_status
FROM members AS m
JOIN class_bookings AS b ON b.member_id = m.member_id
JOIN gym_classes AS c ON c.gym_class_id = b.gym_class_id;
```

first_name,last_name,class_name,starts_at,booking_status
Aida,Karimova,Morning Strength,2026-11-20T04:00:00Z,booked
Daniyar,Omarov,Morning Strength,2026-11-20T04:00:00Z,booked
Madina,Aliyeva,Morning Strength,2026-11-20T04:00:00Z,booked
Timur,Bekov,Morning Strength,2026-11-20T04:00:00Z,booked
Alina,Smailova,Morning Strength,2026-11-20T04:00:00Z,booked
Kamila,Nur,Strength Basics,2026-11-10T04:00:00Z,attended
Daniyar,Omarov,Strength Basics,2026-11-10T04:00:00Z,attended
Aida,Karimova,Strength Basics,2026-11-10T04:00:00Z,attended
Arman,Sultanov,Evening Strength,2026-11-21T13:00:00Z,cancelled
Dana,Lee,Evening Strength,2026-11-21T13:00:00Z,booked


**331 row(s)** returned.

## 4.1 · GROUP BY — sales by plan


In [9]:
show_query("4.1", """SELECT p.plan_name,
       COUNT(ms.membership_id) AS memberships_sold,
       COUNT(DISTINCT m.member_id) AS unique_members,
       COALESCE(SUM(ms.paid_amount_kzt), 0) AS revenue_kzt
FROM membership_plans AS p
LEFT JOIN memberships AS ms ON ms.membership_plan_id = p.membership_plan_id
LEFT JOIN members AS m ON m.member_id = ms.member_id
GROUP BY p.membership_plan_id, p.plan_name;""")


```sql
SELECT p.plan_name,
       COUNT(ms.membership_id) AS memberships_sold,
       COUNT(DISTINCT m.member_id) AS unique_members,
       COALESCE(SUM(ms.paid_amount_kzt), 0) AS revenue_kzt
FROM membership_plans AS p
LEFT JOIN memberships AS ms ON ms.membership_plan_id = p.membership_plan_id
LEFT JOIN members AS m ON m.member_id = ms.member_id
GROUP BY p.membership_plan_id, p.plan_name;
```

plan_name,memberships_sold,unique_members,revenue_kzt
Annual Plus,10,10,"1,754,000.00"
Family,21,12,"1,119,200.00"
Legacy,1,1,"15,000.00"
Monthly,14,12,"258,000.00"
Off-Peak,1,1,"15,000.00"
Premium,11,11,"1,934,000.00"
Quarterly,23,13,"1,227,200.00"
Student,11,11,"130,000.00"
Weekend,13,11,"238,000.00"
Youth,10,10,"118,000.00"


**10 row(s)** returned.

## 4.2 · GROUP BY — activity by trainer


In [10]:
show_query("4.2", """SELECT t.first_name, t.last_name,
       COUNT(DISTINCT c.gym_class_id) AS class_count,
       COUNT(b.member_id) FILTER (
           WHERE b.booking_status IN ('booked', 'attended')
       ) AS non_cancelled_bookings
FROM trainers AS t
LEFT JOIN gym_classes AS c ON c.trainer_id = t.trainer_id
LEFT JOIN class_bookings AS b ON b.gym_class_id = c.gym_class_id
GROUP BY t.trainer_id, t.first_name, t.last_name;""")


```sql
SELECT t.first_name, t.last_name,
       COUNT(DISTINCT c.gym_class_id) AS class_count,
       COUNT(b.member_id) FILTER (
           WHERE b.booking_status IN ('booked', 'attended')
       ) AS non_cancelled_bookings
FROM trainers AS t
LEFT JOIN gym_classes AS c ON c.trainer_id = t.trainer_id
LEFT JOIN class_bookings AS b ON b.gym_class_id = c.gym_class_id
GROUP BY t.trainer_id, t.first_name, t.last_name;
```

first_name,last_name,class_count,non_cancelled_bookings
Askar,Sadykov,3,12
Aigerim,Tursyn,2,7
Marat,Ivanov,2,3
Elena,Petrova,1,1
Victor,Chen,0,0
Dinara,Abenova,3,12
Roman,Sokolov,2,7
Yernar,Bulatov,2,3
Malika,Tulegenova,1,1
Denis,Orlov,0,0


**14 row(s)** returned.

## 5.1 · HAVING — plans purchased by at least two members


In [11]:
show_query("5.1", """SELECT p.plan_name, COUNT(DISTINCT m.member_id) AS unique_members,
       SUM(ms.paid_amount_kzt) AS revenue_kzt
FROM membership_plans AS p
JOIN memberships AS ms ON ms.membership_plan_id = p.membership_plan_id
JOIN members AS m ON m.member_id = ms.member_id
GROUP BY p.membership_plan_id, p.plan_name
HAVING COUNT(DISTINCT m.member_id) >= 2;""")


```sql
SELECT p.plan_name, COUNT(DISTINCT m.member_id) AS unique_members,
       SUM(ms.paid_amount_kzt) AS revenue_kzt
FROM membership_plans AS p
JOIN memberships AS ms ON ms.membership_plan_id = p.membership_plan_id
JOIN members AS m ON m.member_id = ms.member_id
GROUP BY p.membership_plan_id, p.plan_name
HAVING COUNT(DISTINCT m.member_id) >= 2;
```

plan_name,unique_members,revenue_kzt
Monthly,12,"258,000.00"
Quarterly,13,"1,227,200.00"
Student,11,"130,000.00"
Premium,11,"1,934,000.00"
Weekend,11,"238,000.00"
Family,12,"1,119,200.00"
Youth,10,"118,000.00"
Annual Plus,10,"1,754,000.00"


**8 row(s)** returned.

## 5.2 · HAVING — trainers with at least five valid bookings


In [12]:
show_query("5.2", """SELECT t.first_name, t.last_name, COUNT(*) AS non_cancelled_bookings
FROM trainers AS t
JOIN gym_classes AS c ON c.trainer_id = t.trainer_id
JOIN class_bookings AS b ON b.gym_class_id = c.gym_class_id
WHERE b.booking_status <> 'cancelled'
GROUP BY t.trainer_id, t.first_name, t.last_name
HAVING COUNT(*) >= 5;""")


```sql
SELECT t.first_name, t.last_name, COUNT(*) AS non_cancelled_bookings
FROM trainers AS t
JOIN gym_classes AS c ON c.trainer_id = t.trainer_id
JOIN class_bookings AS b ON b.gym_class_id = c.gym_class_id
WHERE b.booking_status <> 'cancelled'
GROUP BY t.trainer_id, t.first_name, t.last_name
HAVING COUNT(*) >= 5;
```

first_name,last_name,non_cancelled_bookings
Askar,Sadykov,12
Aigerim,Tursyn,7
Dinara,Abenova,12
Roman,Sokolov,7
Azamat,Bekenov,62
Yulia,Morozova,62
Nikita,Smirnov,63
Asem,Tulegen,62


**8 row(s)** returned.

## 6.1 · ORDER BY — chronological class schedule


In [13]:
show_query("6.1", """SELECT c.class_name, c.starts_at, c.ends_at,
       t.first_name AS trainer_first_name, t.last_name AS trainer_last_name
FROM gym_classes AS c
JOIN trainers AS t ON t.trainer_id = c.trainer_id
ORDER BY c.starts_at ASC, c.gym_class_id ASC;""")


```sql
SELECT c.class_name, c.starts_at, c.ends_at,
       t.first_name AS trainer_first_name, t.last_name AS trainer_last_name
FROM gym_classes AS c
JOIN trainers AS t ON t.trainer_id = c.trainer_id
ORDER BY c.starts_at ASC, c.gym_class_id ASC;
```

class_name,starts_at,ends_at,trainer_first_name,trainer_last_name
Strength Circuit,2026-11-05T09:00:00Z,2026-11-05T10:00:00Z,Azamat,Bekenov
Yoga Basics,2026-11-05T09:00:00Z,2026-11-05T10:00:00Z,Yulia,Morozova
Cardio Training,2026-11-05T09:00:00Z,2026-11-05T10:00:00Z,Nikita,Smirnov
Pilates Core,2026-11-05T09:00:00Z,2026-11-05T10:00:00Z,Asem,Tulegen
Strength Circuit,2026-11-06T09:00:00Z,2026-11-06T10:00:00Z,Azamat,Bekenov
Yoga Basics,2026-11-06T09:00:00Z,2026-11-06T10:00:00Z,Yulia,Morozova
Cardio Training,2026-11-06T09:00:00Z,2026-11-06T10:00:00Z,Nikita,Smirnov
Pilates Core,2026-11-06T09:00:00Z,2026-11-06T10:00:00Z,Asem,Tulegen
Strength Circuit,2026-11-07T09:00:00Z,2026-11-07T10:00:00Z,Azamat,Bekenov
Yoga Basics,2026-11-07T09:00:00Z,2026-11-07T10:00:00Z,Yulia,Morozova


**40 row(s)** returned.

## 6.2 · ORDER BY — largest membership payments first


In [14]:
show_query("6.2", """SELECT m.first_name, m.last_name, ms.starts_on, ms.paid_amount_kzt
FROM memberships AS ms
JOIN members AS m ON m.member_id = ms.member_id
ORDER BY ms.paid_amount_kzt DESC, m.last_name ASC, m.first_name ASC,
         ms.membership_id ASC;""")


```sql
SELECT m.first_name, m.last_name, ms.starts_on, ms.paid_amount_kzt
FROM memberships AS ms
JOIN members AS m ON m.member_id = ms.member_id
ORDER BY ms.paid_amount_kzt DESC, m.last_name ASC, m.first_name ASC,
         ms.membership_id ASC;
```

first_name,last_name,starts_on,paid_amount_kzt
Diana,Abishev,2026-11-01,"180,000.00"
Ibrahim,Abishev,2026-11-01,"180,000.00"
Maksat,Abishev,2026-11-01,"180,000.00"
Ali,Baimurat,2026-11-01,"180,000.00"
Ermek,Baimurat,2026-11-01,"180,000.00"
Nazym,Baimurat,2026-11-01,"180,000.00"
Zhanel,Baimurat,2026-11-01,"180,000.00"
Asel,Daulet,2026-11-01,"180,000.00"
Farida,Daulet,2026-11-01,"180,000.00"
Karim,Daulet,2026-11-01,"180,000.00"


**115 row(s)** returned.

## 7.1 · Comparison predicates — affordable active plans


In [15]:
show_query("7.1", """SELECT plan_name, price_kzt, duration_days
FROM membership_plans
WHERE is_active = true
  AND price_kzt > 0
  AND price_kzt <= 20000;""")


```sql
SELECT plan_name, price_kzt, duration_days
FROM membership_plans
WHERE is_active = true
  AND price_kzt > 0
  AND price_kzt <= 20000;
```

plan_name,price_kzt,duration_days
Monthly,"20,000.00",30
Student,"12,000.00",30
Weekend,"20,000.00",30
Youth,"12,000.00",30


**4 row(s)** returned.

## 7.2 · Comparison predicates — higher-priced non-legacy plans


In [16]:
show_query("7.2", """SELECT plan_name, price_kzt, duration_days
FROM membership_plans
WHERE plan_name <> 'Legacy'
  AND price_kzt >= 20000;""")


```sql
SELECT plan_name, price_kzt, duration_days
FROM membership_plans
WHERE plan_name <> 'Legacy'
  AND price_kzt >= 20000;
```

plan_name,price_kzt,duration_days
Monthly,"20,000.00",30
Quarterly,"54,000.00",90
Premium,"180,000.00",365
Weekend,"20,000.00",30
Family,"54,000.00",90
Annual Plus,"180,000.00",365


**6 row(s)** returned.

## 8.1 · IN — selected plan types


In [17]:
show_query("8.1", """SELECT plan_name, duration_days, price_kzt
FROM membership_plans
WHERE plan_name IN ('Student', 'Monthly', 'Quarterly');""")


```sql
SELECT plan_name, duration_days, price_kzt
FROM membership_plans
WHERE plan_name IN ('Student', 'Monthly', 'Quarterly');
```

plan_name,duration_days,price_kzt
Monthly,30,"20,000.00"
Quarterly,90,"54,000.00"
Student,30,"12,000.00"


**3 row(s)** returned.

## 8.2 · IN — attended or cancelled bookings


In [18]:
show_query("8.2", """SELECT member_id, gym_class_id, booking_status
FROM class_bookings
WHERE booking_status IN ('attended', 'cancelled');""")


```sql
SELECT member_id, gym_class_id, booking_status
FROM class_bookings
WHERE booking_status IN ('attended', 'cancelled');
```

member_id,gym_class_id,booking_status
12,3,attended
2,3,attended
1,3,attended
11,4,cancelled
12,5,attended
6,5,attended
1,5,attended
14,6,cancelled
13,8,cancelled
24,11,attended


**172 row(s)** returned.

## 9.1 · EXISTS — members with a valid membership


In [19]:
show_query("9.1", """SELECT m.member_id, m.first_name, m.last_name
FROM members AS m
WHERE EXISTS (
    SELECT 1
    FROM memberships AS ms
    WHERE ms.member_id = m.member_id
      AND ms.starts_on <= '2026-11-12'
      AND ms.ends_on > '2026-11-12'
);""")


```sql
SELECT m.member_id, m.first_name, m.last_name
FROM members AS m
WHERE EXISTS (
    SELECT 1
    FROM memberships AS ms
    WHERE ms.member_id = m.member_id
      AND ms.starts_on <= '2026-11-12'
      AND ms.ends_on > '2026-11-12'
);
```

member_id,first_name,last_name
1,Aida,Karimova
2,Daniyar,Omarov
6,Alina,Smailova
7,Timur,Bekov
8,Madina,Aliyeva
10,Dana,Lee
11,Arman,Sultanov
12,Kamila,Nur
16,Aruzhan,Beketova
17,Miras,Akhmetov


**84 row(s)** returned.

## 9.2 · NOT EXISTS — members with no bookings


In [20]:
show_query("9.2", """SELECT m.member_id, m.first_name, m.last_name
FROM members AS m
WHERE NOT EXISTS (
    SELECT 1
    FROM class_bookings AS b
    WHERE b.member_id = m.member_id
);""")


```sql
SELECT m.member_id, m.first_name, m.last_name
FROM members AS m
WHERE NOT EXISTS (
    SELECT 1
    FROM class_bookings AS b
    WHERE b.member_id = m.member_id
);
```

member_id,first_name,last_name
9,Ruslan,Kim
15,Nurlan,Asan
21,Ilyas,Rakhimov
27,Sanzhar,Ospanov
28,Ali,Abishev
41,Oleg,Abishev
54,Ibrahim,Baimurat
67,Asel,Daulet
80,Raushan,Daulet
93,Zhanel,Erbol


**10 row(s)** returned.

## 10.1 · Aggregates — purchase count and total payments


In [21]:
show_query("10.1", """SELECT COUNT(*) AS total_memberships,
       SUM(paid_amount_kzt) AS total_revenue_kzt
FROM memberships;""")


```sql
SELECT COUNT(*) AS total_memberships,
       SUM(paid_amount_kzt) AS total_revenue_kzt
FROM memberships;
```

total_memberships,total_revenue_kzt
115,"6,808,400.00"


**1 row(s)** returned.

## 10.2 · Aggregates — active-plan price statistics


In [22]:
show_query("10.2", """SELECT ROUND(AVG(price_kzt), 2) AS average_price_kzt,
       MIN(price_kzt) AS minimum_price_kzt,
       MAX(price_kzt) AS maximum_price_kzt
FROM membership_plans
WHERE is_active = true;""")


```sql
SELECT ROUND(AVG(price_kzt), 2) AS average_price_kzt,
       MIN(price_kzt) AS minimum_price_kzt,
       MAX(price_kzt) AS maximum_price_kzt
FROM membership_plans
WHERE is_active = true;
```

average_price_kzt,minimum_price_kzt,maximum_price_kzt
"66,500.00","12,000.00","180,000.00"


**1 row(s)** returned.

## 11 · DELETE — remove an unused demo member


In [23]:
display(Markdown("**Before the change**"))
show_query("11-before", "SELECT member_id, first_name, last_name, email FROM members WHERE email = 'nurlan@example.test';")
with rollback_demo():
    display(Markdown("**Change inside the transaction**"))
    show_query("11", """DELETE FROM members
WHERE email = 'nurlan@example.test'
RETURNING member_id, first_name, last_name, email;""")
display(Markdown("**After rollback — original data restored**"))
show_query("11-after", "SELECT member_id, first_name, last_name, email FROM members WHERE email = 'nurlan@example.test';")


**Before the change**

```sql
SELECT member_id, first_name, last_name, email FROM members WHERE email = 'nurlan@example.test';
```

member_id,first_name,last_name,email
15,Nurlan,Asan,nurlan@example.test


**1 row(s)** returned.

**Change inside the transaction**

```sql
DELETE FROM members
WHERE email = 'nurlan@example.test'
RETURNING member_id, first_name, last_name, email;
```

... (2 outputs left out)

**After rollback — original data restored**

```sql
SELECT member_id, first_name, last_name, email FROM members WHERE email = 'nurlan@example.test';
```

member_id,first_name,last_name,email
15,Nurlan,Asan,nurlan@example.test


**1 row(s)** returned.

## 12 · UPDATE — change a plan price


In [24]:
display(Markdown("**Before the change**"))
show_query("12-before", "SELECT plan_name, price_kzt FROM membership_plans WHERE plan_name = 'Monthly';")
with rollback_demo():
    display(Markdown("**Change inside the transaction**"))
    show_query("12", """UPDATE membership_plans
SET price_kzt = 22000.00
WHERE plan_name = 'Monthly'
RETURNING membership_plan_id, plan_name, price_kzt;""")
display(Markdown("**After rollback — original data restored**"))
show_query("12-after", "SELECT plan_name, price_kzt FROM membership_plans WHERE plan_name = 'Monthly';")


**Before the change**

```sql
SELECT plan_name, price_kzt FROM membership_plans WHERE plan_name = 'Monthly';
```

plan_name,price_kzt
Monthly,"20,000.00"


**1 row(s)** returned.

**Change inside the transaction**

```sql
UPDATE membership_plans
SET price_kzt = 22000.00
WHERE plan_name = 'Monthly'
RETURNING membership_plan_id, plan_name, price_kzt;
```

... (2 outputs left out)

**After rollback — original data restored**

```sql
SELECT plan_name, price_kzt FROM membership_plans WHERE plan_name = 'Monthly';
```

plan_name,price_kzt
Monthly,"20,000.00"


**1 row(s)** returned.

## Summary


In [25]:
labels = ["1.1", "1.2", "2.1", "2.2", "3.1", "3.2", "4.1", "4.2",
          "5.1", "5.2", "6.1", "6.2", "7.1", "7.2", "8.1", "8.2",
          "9.1", "9.2", "10.1", "10.2", "11", "12"]
display_table(["example", "rows_returned"], [(label, result_counts[label]) for label in labels])
assert all(result_counts[label] > 0 for label in labels), "Restore the provided demo database and rerun."
assert conn.execute("SELECT COUNT(*) FROM members WHERE email = 'nurlan@example.test'").fetchone()[0] == 1
assert conn.execute("SELECT price_kzt FROM membership_plans WHERE plan_name = 'Monthly'").fetchone()[0] == 20000
conn.close()
print("All 22 examples completed. DELETE and UPDATE were rolled back. Connection closed.")


example,rows_returned
1.1,100
1.2,8
2.1,69
2.2,40
3.1,115
3.2,331
4.1,10
4.2,14
5.1,8
5.2,8


All 22 examples completed. DELETE and UPDATE were rolled back. Connection closed.
